In [1]:
import os
import shutil

# Tes datasets sources
face_root = "face_dataset"
plate_root = "plate_dataset"
out_root = "blur_dataset"

splits = ["train", "valid", "test"]  # inclut aussi test maintenant

# Mapping des classes
# Face = 0
# License_plate = 1
class_mapping = {
    "face": 0,
    "license_plate": 1
}

# Créer la nouvelle structure
for split in splits:
    os.makedirs(os.path.join(out_root, "images", split), exist_ok=True)
    os.makedirs(os.path.join(out_root, "labels", split), exist_ok=True)

def process_dataset(src_root, split, cls_id):
    """Copie images + labels en changeant l'ID de classe dans les fichiers .txt"""
    img_dir = os.path.join(src_root, split, "images")
    lbl_dir = os.path.join(src_root, split, "labels")

    for fname in os.listdir(img_dir):
        if not fname.endswith((".jpg", ".png", ".jpeg")):
            continue
        
        # Copier image
        src_img = os.path.join(img_dir, fname)
        dst_img = os.path.join(out_root, "images", split, fname)
        shutil.copy(src_img, dst_img)

        # Copier et réécrire label
        lbl_name = os.path.splitext(fname)[0] + ".txt"
        src_lbl = os.path.join(lbl_dir, lbl_name)
        dst_lbl = os.path.join(out_root, "labels", split, lbl_name)

        if os.path.exists(src_lbl):
            with open(src_lbl, "r") as f:
                lines = f.readlines()

            # Remplacer toutes les classes par le bon ID
            new_lines = []
            for line in lines:
                parts = line.strip().split()
                parts[0] = str(cls_id)  # remplacer la classe par celle qu'on veut
                new_lines.append(" ".join(parts) + "\n")

            with open(dst_lbl, "w") as f:
                f.writelines(new_lines)

# Appliquer fusion
for split in splits:
    process_dataset(face_root, split, class_mapping["face"])
    process_dataset(plate_root, split, class_mapping["license_plate"])

print("✅ Fusion terminée : blur_dataset prêt !")


✅ Fusion terminée : blur_dataset prêt !


In [7]:
from ultralytics import YOLO
import cv2

def blur_region(image, x1, y1, x2, y2):
    """Applique un flou gaussien sur une région de l'image."""
    roi = image[y1:y2, x1:x2]
    if roi.size == 0:
        return image
    roi_blur = cv2.GaussianBlur(roi, (51, 51), 30)
    image[y1:y2, x1:x2] = roi_blur
    return image

def anonymize_image(model_path, image_path, save_path="output.jpg", conf=0.4):
    """Détecte et floute faces et plaques sur une image."""
    model = YOLO(model_path)
    image = cv2.imread(image_path)
    if image is None:
        print(f"[ERROR] Impossible de lire l'image : {image_path}")
        return

    results = model.predict(image, conf=conf)

    for r in results:
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            image = blur_region(image, x1, y1, x2, y2)

    cv2.imwrite(save_path, image)
    print(f"[INFO] Image anonymisée sauvegardée : {save_path}")

# Exemple d'utilisation directement dans le notebook
anonymize_image("blur.pt", "voiture.jpg", "voiture_floutee.jpg", conf=0.4)



0: 448x640 1 face, 14.0ms
Speed: 2.3ms preprocess, 14.0ms inference, 1.4ms postprocess per image at shape (1, 3, 448, 640)
[INFO] Image anonymisée sauvegardée : voiture_floutee.jpg
